### 0. Setup

In [ ]:
#@title Reproducibility

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found — training will be slow.")
    print("    Runtime → Change runtime type → T4 GPU")

Device : cuda
GPU    : NVIDIA L4


In [ ]:
#@title Downloads

!pip install -q --upgrade tqdm
!pip install midvoxio
!pip install onnx onnxscript onnxruntime
!pip install -U onnxruntime
!pip install lpips
!pip install onnx

In [ ]:
#@title Libraries
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from midvoxio.voxio import vox_to_arr
from midvoxio.voxio import viz_vox
from midvoxio.voxio import write_list_to_vox
from midvoxio.voxio import plot_3d
from mpl_toolkits.mplot3d import Axes3D
from google.colab import drive
import scipy
import torch.optim as optim
import random
from tqdm import tqdm
import lpips
import onnx
import onnxscript


In [ ]:
#@title Download/External links
!wget -O Regular_tree.vox "https://voxbox.store/api/model/download?id=8ryE8w3LOy"
!wget -O Fall_tree.vox "https://voxbox.store/api/model/download?id=rC2Fnv8YI2"
!wget -O Chicken.vox "https://voxbox.store/api/model/download?id=nTqJDfvpU9"
!wget -O Pine_tree.vox "https://voxbox.store/api/model/download?id=XHxA65ZDGj"
!wget -O Furnace.vox "https://voxbox.store/api/model/download?id=ei6SEQJPSs"
!wget -O Mug.vox "https://voxbox.store/api/model/download?id=d4cA4CJFYp"
!wget -O Mushroom.vox "https://voxbox.store/api/model/download?id=ky7oAEUGsh"
!wget -O Globe.vox "https://voxbox.store/api/model/download?id=05PJncdAqK"

--2026-04-17 18:36:55--  https://voxbox.store/api/model/download?id=8ryE8w3LOy
Resolving voxbox.store (voxbox.store)... 5.161.231.181, 2a01:4ff:f0:8ec9::1
Connecting to voxbox.store (voxbox.store)|5.161.231.181|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1740611 (1.7M) [application/x-authorware-bin]
Saving to: ‘Regular_tree.vox’

Regular_tree.vox    100%[===================>]   1.66M  1.40MB/s    in 1.2s    

2026-04-17 18:36:58 (1.40 MB/s) - ‘Regular_tree.vox’ saved [1740611/1740611]

--2026-04-17 18:36:58--  https://voxbox.store/api/model/download?id=rC2Fnv8YI2
Resolving voxbox.store (voxbox.store)... 5.161.231.181, 2a01:4ff:f0:8ec9::1
Connecting to voxbox.store (voxbox.store)|5.161.231.181|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1740505 (1.7M) [application/x-authorware-bin]
Saving to: ‘Fall_tree.vox’

Fall_tree.vox       100%[===================>]   1.66M  1.45MB/s    in 1.1s    

2026-04-17 18:37:00 (1.45 MB/s) - ‘Fall

### 1. Perception

In [ ]:
class Perception(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.perceive = nn.Conv3d(
            channels,
            channels * 4,       # identity + sobel_x + sobel_y + sobel_z
            kernel_size=3,
            padding=1,
            groups=channels,
            bias=False
        )

        
        identity = torch.zeros(3, 3, 3, dtype=torch.float32)
        identity[1, 1, 1] = 1.0


        kernels = torch.stack([identity, sobel_x, sobel_y, laplace], dim=0)
        kernels = kernels.unsqueeze(1)
        kernels = kernels.repeat(channels, 1, 1, 1, 1)

        with torch.no_grad():
            self.perceive.weight.copy_(kernels)

        self.n_channels = channels

    def forward(self, x):
        return self.perceive(x)